# 02 · Joins & Window Functions

Multi-table joins and window functions on enriched sales.

`sales` here is already typed and cleaned for you via `read_sales_typed` (you
built that by hand in notebook 03's territory). The store/category text is
normalized in setup so grouping behaves — text cleaning is covered in
notebook 03.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "utils" / "spark.py").exists() and _root != _root.parent:
    _root = _root.parent
for _p in (str(_root), str(_root / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from utils.spark import build_session, read_dim, read_sales_raw, read_sales_typed

spark = build_session("nb02")
print("Spark", spark.version, "ready")

sales = read_sales_typed(spark)
# Normalize dimension text up front so groupBy keys collapse cleanly.
# (The casing/whitespace cleaning technique itself is taught in notebook 03.)
products = read_dim(spark, "dim_products").withColumn("category", F.initcap(F.trim("category")))
stores = read_dim(spark, "dim_stores").withColumn("region", F.initcap(F.trim("region")))
customers = read_dim(spark, "dim_customers")
print("typed sales rows:", sales.count())

## Task 1: Enrich sales with dimensions

Inner-join `sales` to `products` (on `product_id`) and `stores` (on `store_id`) to add `category`, `brand`, `region`, `channel`. Name the result `enriched`.

Note: some sales reference non-existent products (orphan FKs); an inner join drops them, which is fine here.

In [ ]:
enriched = (
    sales
    .join(products.select("product_id", "category", "brand"), "product_id")
    .join(stores.select("store_id", "region", "channel"), "store_id")
)
enriched.select("txn_id", "category", "brand", "region", "channel", "revenue").show(5)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
for _c in ["category", "brand", "region", "channel"]:
    assert _c in enriched.columns, _c
assert enriched.count() <= sales.count()  # inner join may drop orphans
print("✅ Task 1 passed — enriched rows:", enriched.count())

## Task 2: Top 3 brands per region

Using `enriched`, total revenue per `(region, brand)`, then rank brands within each region by revenue and keep the top 3. Name it `top_brands` with columns `region`, `brand`, `brand_revenue`, `rnk`. Use a window + `row_number`.

In [ ]:
_rev = enriched.groupBy("region", "brand").agg(
    F.round(F.sum("revenue"), 2).alias("brand_revenue")
)
_w = Window.partitionBy("region").orderBy(F.desc("brand_revenue"))
top_brands = (
    _rev.withColumn("rnk", F.row_number().over(_w))
    .filter(F.col("rnk") <= 3)
    .orderBy("region", "rnk")
)
top_brands.show()

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert {"region", "brand", "brand_revenue", "rnk"} <= set(top_brands.columns)
assert top_brands.groupBy("region").count().filter("count > 3").count() == 0
print("✅ Task 2 passed")

## Task 3: Broadcast join

Re-do the product join but explicitly **broadcast** the small `products` dimension to avoid a shuffle. Name it `enriched_bc`. (Use `F.broadcast`.)

In [ ]:
enriched_bc = sales.join(
    F.broadcast(products.select("product_id", "category")), "product_id"
)
enriched_bc.show(5)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
_plan = enriched_bc._jdf.queryExecution().executedPlan().toString()
assert "Broadcast" in _plan, "expected a broadcast join in the physical plan"
print("✅ Task 3 passed — broadcast join confirmed")

## Task 4: Rolling 7-day revenue

Aggregate `enriched` to daily total revenue (`daily_revenue`), then add a 7-day trailing sum `rolling_7d` using a window ordered by date with `rowsBetween(-6, 0)`. Name the result `daily` with columns `txn_date`, `daily_revenue`, `rolling_7d`, ordered by date.

In [ ]:
daily = enriched.groupBy("txn_date").agg(
    F.round(F.sum("revenue"), 2).alias("daily_revenue")
)
_w7 = Window.orderBy("txn_date").rowsBetween(-6, 0)
daily = daily.withColumn(
    "rolling_7d", F.round(F.sum("daily_revenue").over(_w7), 2)
).orderBy("txn_date")
daily.show(10)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert {"txn_date", "daily_revenue", "rolling_7d"} <= set(daily.columns)
_rows = daily.orderBy("txn_date").limit(7).collect()
_exp = round(sum(r["daily_revenue"] for r in _rows), 2)
assert abs(_rows[6]["rolling_7d"] - _exp) < 1.0, (_rows[6]["rolling_7d"], _exp)
print("✅ Task 4 passed")

## Task 5: Week-over-week growth (lag)

From `enriched`, compute weekly revenue. Build a `year_week` key as `concat_ws("-W", year(txn_date), lpad(weekofyear(txn_date), 2, "0"))` (e.g. `2026-W23`). Then use `lag` to get the previous week's revenue and a `wow_pct` growth percentage. Name it `wow` with columns `year_week`, `weekly_revenue`, `prev_week`, `wow_pct`, ordered by week.

In [ ]:
_year_week = F.concat_ws(
    "-W", F.year("txn_date"), F.lpad(F.weekofyear("txn_date").cast("string"), 2, "0")
)
_weekly = (
    enriched.withColumn("year_week", _year_week)
    .groupBy("year_week")
    .agg(F.round(F.sum("revenue"), 2).alias("weekly_revenue"))
)
_wl = Window.orderBy("year_week")
wow = (
    _weekly.withColumn("prev_week", F.lag("weekly_revenue").over(_wl))
    .withColumn(
        "wow_pct",
        F.round((F.col("weekly_revenue") - F.col("prev_week")) / F.col("prev_week") * 100, 2),
    )
    .orderBy("year_week")
)
wow.show()

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert {"year_week", "weekly_revenue", "prev_week", "wow_pct"} <= set(wow.columns)
assert wow.orderBy("year_week").first()["prev_week"] is None  # first week has no previous
print("✅ Task 5 passed")

## Task 6: Customer RFM scoring

For sales with a non-null `customer_id`, compute per customer: `recency_days` (days from last purchase to 2026-06-05), `frequency` (#txns), `monetary` (total revenue). Then add `m_quartile` = `ntile(4)` over monetary. Name it `rfm` with columns `customer_id`, `recency_days`, `frequency`, `monetary`, `m_quartile`. (Use `sales`, which still has `customer_id`.)

In [ ]:
_ref_date = F.lit("2026-06-05").cast("date")
_base = (
    sales.filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id")
    .agg(
        F.max("txn_date").alias("_last"),
        F.count("*").alias("frequency"),
        F.round(F.sum("revenue"), 2).alias("monetary"),
    )
)
rfm = (
    _base.withColumn("recency_days", F.datediff(_ref_date, F.col("_last")))
    .withColumn("m_quartile", F.ntile(4).over(Window.orderBy("monetary")))
    .select("customer_id", "recency_days", "frequency", "monetary", "m_quartile")
)
rfm.show(5)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert {"customer_id", "recency_days", "frequency", "monetary", "m_quartile"} <= set(rfm.columns)
assert rfm.select("m_quartile").distinct().count() == 4
assert rfm.filter(F.col("recency_days") < 0).count() == 0
print("✅ Task 6 passed")

---
🎉 **Solution notebook** — all cells should run top to bottom.